# Phase 6 — Redundancy Reduction (CD-HIT Clustering)

Phase 5 produced the hand-curated ortholog set (`manually_curated_orthologs.fasta`)
with `Genus|Accession|Species` headers. This phase reduces sequence redundancy by
clustering that set with **CD-HIT** at four identity thresholds, and verifies that
clustering does not silently drop any species.

The thresholds serve different downstream purposes:

- **99%** — removes only near-duplicate strain variants.
- **98%** — the reference set for the alignment (Phase 7): near-identical strains
  collapse, but species-level diversity is retained.
- **95%** — a more aggressively reduced set.
- **90%** — a broad, genus-level backbone.

Generating all four also supports a redundancy-sensitivity check: a downstream
result stable across thresholds is not an artifact of sequence redundancy.

Because CD-HIT clusters by sequence *identity* rather than taxonomy, two distinct
species with near-identical A29L can collapse into one representative. The
species-coverage step makes that visible per threshold, so the choice of
reference set is verified rather than assumed.

**Reference hold-out.** The reference monkeypox sequence(s) in
`hold_out_accessions` are removed *before* clustering and added back to every
non-redundant set afterward. This guarantees they are never lost to a cluster —
so the reference is present in every threshold set and in all Phase 7 figures —
while keeping the process fully reproducible (nothing is hand-edited).

**Steps**

0. Setup & configuration
1. Cluster at each identity threshold
2. Summarize clustering across thresholds
3. Species-coverage check per threshold

## Step 0 — Setup & configuration

Clustering thresholds are collected in `cdhit_identities`. **CD-HIT word length
(`-n`)** must match the identity threshold; for proteins, `-n 5` covers 0.7–1.0,
so it is valid for all four thresholds here. `cdhit_word_length` derives the
correct `-n` from each identity automatically. Genus/species come from the
`Genus|Accession|Species` headers written in Phase 5, so this phase makes no
network calls. External tools run through the shared `run_command` helper.

In [1]:
from __future__ import annotations

import logging
import subprocess
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path

from seqio_utils import parse_fasta_records, count_cdhit_clusters

In [2]:
@dataclass(frozen=True)
class Config:
    curated_fasta: Path = Path("output_phase5/manually_curated_orthologs.fasta")

    wsl_prefix: list[str] = field(default_factory=lambda: ["wsl"])
    cdhit_max_memory_mb: int = 2000
    cdhit_identities: tuple[float, ...] = (0.99, 0.98, 0.95, 0.90)
    reference_identity: float = 0.98
    hold_out_accessions: tuple[str, ...] = ("YFT67316.1", "YFT66960.1")

    out_dir: Path = field(default_factory=lambda: Path("output_phase6"))

    @property
    def clustering_input(self) -> Path:
        return self.out_dir / "clustering_input.fasta"

    def nr_fasta(self, identity: float) -> Path:
        return self.out_dir / f"orthologs_nr{round(identity * 100)}.fasta"

    @property
    def reference_fasta(self) -> Path:
        return self.nr_fasta(self.reference_identity)


def cdhit_word_length(identity: float) -> int:
    if identity >= 0.7:
        return 5
    if identity >= 0.6:
        return 4
    if identity >= 0.5:
        return 3
    return 2


CFG = Config()
CFG.out_dir.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(CFG.out_dir / "phase6.log"),
    ],
)
log = logging.getLogger("phase6")

if not CFG.curated_fasta.exists():
    raise FileNotFoundError(
        f"{CFG.curated_fasta} not found — run Phase 5 first."
    )

In [3]:
class CommandError(RuntimeError):
    """Raised when an external tool exits with a non-zero status."""


def run_command(cmd: list[str], log_file: Path) -> None:
    log.info("Running: %s", " ".join(cmd))
    with open(log_file, "w", encoding="utf-8") as fh:
        result = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT)
    if result.returncode != 0:
        raise CommandError(
            f"Command failed (exit {result.returncode}): {' '.join(cmd)}. "
            f"See {log_file} for details."
        )

## Step 1 — Cluster at each identity threshold

The curated set is first split: sequences whose accession is in
`hold_out_accessions` (the reference monkeypox sequences) are set aside, and
CD-HIT clusters only the remainder, written to `clustering_input.fasta`. After
each run, the held-out references are appended to that threshold's output, so
they appear in every non-redundant set regardless of how aggressively the rest
is clustered. Each output FASTA is checkpointed; the held-out sequences are
identified by accession (the middle field of the `Genus|Accession|Species`
header).

In [4]:
def run_cdhit(input_fasta: Path, output_fasta: Path, identity: float, cfg: Config = CFG) -> None:
    log.info("Running CD-HIT at %.0f%% identity ...", identity * 100)
    run_command(
        cfg.wsl_prefix + [
            "cd-hit",
            "-i", input_fasta.as_posix(), "-o", output_fasta.as_posix(),
            "-c", str(identity), "-n", str(cdhit_word_length(identity)),
            "-d", "0", "-M", str(cfg.cdhit_max_memory_mb),
        ],
        log_file=output_fasta.with_suffix(".cdhit.log"),
    )


def acc_of(hdr: str) -> str:
    parts = hdr.split("|")
    return parts[1] if len(parts) >= 2 else hdr.split()[0]


curated = parse_fasta_records(CFG.curated_fasta)
hold_out = [(h, s) for h, s in curated if acc_of(h) in CFG.hold_out_accessions]
to_cluster = [(h, s) for h, s in curated if acc_of(h) not in CFG.hold_out_accessions]

found = {acc_of(h) for h, _ in hold_out}
missing_holdouts = [a for a in CFG.hold_out_accessions if a not in found]
if missing_holdouts:
    log.warning("Held-out accession(s) not present in curated set: %s", missing_holdouts)
log.info("Holding %d reference sequence(s) out of clustering: %s",
         len(hold_out), sorted(found))

with open(CFG.clustering_input, "w", encoding="utf-8") as out:
    for hdr, seq in to_cluster:
        out.write(f">{hdr}\n{seq}\n")

for identity in CFG.cdhit_identities:
    out_fasta = CFG.nr_fasta(identity)
    if out_fasta.exists():
        log.info("%s exists — skipping CD-HIT at %.0f%%.", out_fasta, identity * 100)
        continue
    run_cdhit(CFG.clustering_input, out_fasta, identity)
    with open(out_fasta, "a", encoding="utf-8") as out:
        for hdr, seq in hold_out:
            out.write(f">{hdr}\n{seq}\n")
    log.info("DONE — %s (%d clustered + %d held-out references)",
             out_fasta, sum(1 for _ in parse_fasta_records(out_fasta)) - len(hold_out), len(hold_out))

n_curated = len(curated)
log.info("Curated input: %d sequences (%d clustered, %d held out).",
         n_curated, len(to_cluster), len(hold_out))

2026-07-23 01:15:17,614 [INFO] Holding 2 reference sequence(s) out of clustering: ['YFT66960.1', 'YFT67316.1']
2026-07-23 01:15:17,619 [INFO] Running CD-HIT at 99% identity ...
2026-07-23 01:15:17,621 [INFO] Running: wsl cd-hit -i output_phase6/clustering_input.fasta -o output_phase6/orthologs_nr99.fasta -c 0.99 -n 5 -d 0 -M 2000
2026-07-23 01:15:19,935 [INFO] DONE — output_phase6\orthologs_nr99.fasta (65 clustered + 2 held-out references)
2026-07-23 01:15:19,938 [INFO] Running CD-HIT at 98% identity ...
2026-07-23 01:15:19,939 [INFO] Running: wsl cd-hit -i output_phase6/clustering_input.fasta -o output_phase6/orthologs_nr98.fasta -c 0.98 -n 5 -d 0 -M 2000
2026-07-23 01:15:20,368 [INFO] DONE — output_phase6\orthologs_nr98.fasta (60 clustered + 2 held-out references)
2026-07-23 01:15:20,370 [INFO] Running CD-HIT at 95% identity ...
2026-07-23 01:15:20,372 [INFO] Running: wsl cd-hit -i output_phase6/clustering_input.fasta -o output_phase6/orthologs_nr95.fasta -c 0.95 -n 5 -d 0 -M 2000
20

## Step 2 — Summarize clustering across thresholds

Reads each CD-HIT `.clstr` file to report how many representative sequences each
threshold produced, alongside the percentage reduction from the curated input —
the table for the methods section.

In [5]:
log.info("Clustering summary (curated input: %d sequences):", n_curated)
log.info("  %-10s %-12s %-10s", "threshold", "clusters", "reduction")

for identity in CFG.cdhit_identities:
    clstr = CFG.nr_fasta(identity).with_suffix(".fasta.clstr")
    n_clusters = count_cdhit_clusters(clstr)
    reduction = (1 - n_clusters / n_curated) * 100 if n_curated else 0.0
    log.info("  %-10s %-12d %.1f%%", f"{round(identity * 100)}%", n_clusters, reduction)

2026-07-23 01:15:21,327 [INFO] Clustering summary (curated input: 76 sequences):
2026-07-23 01:15:21,330 [INFO]   threshold  clusters     reduction 
2026-07-23 01:15:21,333 [INFO]   99%        65           14.5%
2026-07-23 01:15:21,336 [INFO]   98%        60           21.1%
2026-07-23 01:15:21,339 [INFO]   95%        45           40.8%
2026-07-23 01:15:21,342 [INFO]   90%        38           50.0%


## Step 3 — Species-coverage check per threshold

CD-HIT clusters by sequence identity, not taxonomy, so distinct species with
near-identical A29L can collapse into a single representative. The genus and
species are read directly from each `Genus|Accession|Species` header (no network
lookup needed, since Phase 5 already curated them). For every threshold this
reports how many species are represented and — the key output — lists any
species present in the curated set but left with **zero** representatives at that
threshold. Use this to confirm the reference threshold retains at least one
sequence per species.

In [6]:
def species_of(hdr: str) -> str:
    parts = hdr.split("|")
    return parts[2] if len(parts) >= 3 else "unclassified"


curated_species = {species_of(h) for h, _ in parse_fasta_records(CFG.curated_fasta)}
log.info("Curated set spans %d species.", len(curated_species))

for identity in CFG.cdhit_identities:
    reps = parse_fasta_records(CFG.nr_fasta(identity))
    rep_species = defaultdict(int)
    for hdr, _ in reps:
        rep_species[species_of(hdr)] += 1

    represented = set(rep_species)
    missing = sorted(curated_species - represented)

    log.info("%.0f%%: %d representatives cover %d/%d species.",
             identity * 100, len(reps), len(represented), len(curated_species))
    if missing:
        log.warning("%.0f%%: %d species lost (no representative): %s",
                    identity * 100, len(missing), missing)
    else:
        log.info("%.0f%%: every curated species retains at least one representative.", identity * 100)

log.info("Reference set for Phase 7: %s (%.0f%% identity)",
         CFG.reference_fasta, CFG.reference_identity * 100)
log.info("Phase 6 complete. Outputs in %s", CFG.out_dir)

2026-07-23 01:15:21,399 [INFO] Curated set spans 44 species.
2026-07-23 01:15:21,403 [INFO] 99%: 67 representatives cover 44/44 species.
2026-07-23 01:15:21,404 [INFO] 99%: every curated species retains at least one representative.
2026-07-23 01:15:21,407 [INFO] 98%: 62 representatives cover 44/44 species.
2026-07-23 01:15:21,410 [INFO] 98%: every curated species retains at least one representative.
2026-07-23 01:15:21,412 [INFO] 95%: 47 representatives cover 36/44 species.
2026-07-23 01:15:21,413 [WARNING] 95%: 8 species lost (no representative): ['Orthopoxvirus_abatino', 'Orthopoxvirus_akhmetapox', 'Orthopoxvirus_cowpox', 'Orthopoxvirus_ectomelia', 'Orthopoxvirus_taterapox', 'Orthopoxvirus_variola', 'Unclassified_moosepox_virus', 'Unclassified_white-tailed_deer_poxvirus']
2026-07-23 01:15:21,417 [INFO] 90%: 40 representatives cover 34/44 species.
2026-07-23 01:15:21,418 [WARNING] 90%: 10 species lost (no representative): ['Capripoxvirus_lumpyskinpox', 'Orthopoxvirus_abatino', 'Orthop